## Prepare predicted magnitude and redshift dataset

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
import numpy as np
from os import environ
from pathlib import Path
from einops import rearrange
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from hydra import initialize, compose
from hydra.utils import instantiate

from bliss.surveys.dc2 import DC2
from pathlib import Path

home = str(Path().resolve().parents[2])

In [ ]:
with initialize(config_path=f"../../dc2", version_base=None):
    notebook_cfg = compose("notebook_config")

### Load One Full Image

In [ ]:
dc2: DC2 = instantiate(notebook_cfg.surveys.dc2)
test_sample = dc2.get_plotting_sample(0)

In [ ]:
# full image (4000 x 4000)
fig, ax = plt.subplots(figsize=(8, 8))
image = test_sample["image"][0]
ax.imshow(np.log((image - image.min()) + 10), cmap="viridis");

In [ ]:
cur_image_wcs = test_sample["wcs"]
cur_image_true_full_catalog = test_sample["full_catalog"]
cur_image_match_id = test_sample["match_id"]

### Load the trained model and make prediction

In [ ]:
# change this model path according to your training setting
MODEL_PATH = f"{home}/output/DC2_experiments/DC2_psf_aug_asinh/checkpoints/best_encoder.ckpt"
encoder = instantiate(notebook_cfg.encoder)
pretrained_weights = torch.load(MODEL_PATH)["state_dict"]
encoder.load_state_dict(pretrained_weights)
encoder.eval();

In [ ]:
with open(f"{home}/case_studies/dc2/DC2_exp_output/plotting_model_output.pkl", "rb") as inputp: 
    out_dict = pickle.load(inputp)

In [ ]:
out_dict

In [ ]:
bliss_catalog = out_dict["mode_cat"]
bliss_catalog = bliss_catalog.to_full_catalog(4)
# bliss_catalog["plocs"] = bliss_catalog["plocs"] + 4
matcher = instantiate(notebook_cfg.encoder.matcher)
metrics = instantiate(notebook_cfg.encoder.metrics)
matching = matcher.match_catalogs(test_sample["full_catalog"], bliss_catalog)

### Plotting

In [ ]:
# galaxy flux distribution
from einops import rearrange, reduce
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import min_weight_full_bipartite_matching
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 7))
ax.hist(cur_image_true_full_catalog["galaxy_fluxes"][0, :, 2], 
        np.linspace(100, 10000, num=100));

In [ ]:
# flux error
true_matches, bliss_matches = matching[0]
true_flux = cur_image_true_full_catalog.on_fluxes[0, true_matches, :]
bliss_flux = bliss_catalog.on_fluxes[0, bliss_matches, :]
flux_err = true_flux - bliss_flux
est_band_flux = bliss_flux[:, 2]
true_band_flux = true_flux[:, 2]
res = flux_err[:, 2] / true_band_flux
sns.scatterplot(
    x = np.array(true_band_flux), 
    y = np.array(res), 
    alpha = 0.6,
    c = np.log(true_flux[:, 2]), 
    cmap = "viridis"
)
plt.xlim(90, 1000)
plt.ylabel('z-band Flux Residual')
plt.xlabel('Truth z-band Flux')
plt.axhline(y=0, color='r', linestyle='--')
plt.show()

In [ ]:
# redshift
true_redshift = cur_image_true_full_catalog["redshift"][0, true_matches, :]
true_flux = cur_image_true_full_catalog.on_fluxes[0, true_matches, :]
bliss_flux = bliss_catalog.on_fluxes[0, bliss_matches, :]
bliss_flux_nmgy = bliss_flux / 3631
true_flux_nmgy = true_flux / 3631 # transform from nanojansky to nanomaggie

true_mag = 22.5 - 2.5 * torch.log10(true_flux_nmgy)
bliss_mag = 22.5 - 2.5 * torch.log10(bliss_flux_nmgy)

In [ ]:
def transform_column(df):
    """tansform column name to order that meet regression model

    Args:
        df: DataFrame
    
    Returns:
        DataFrame
    """
    column_names = ['mag_g', 'mag_i', 'mag_r', 'mag_u', 'mag_y', 'mag_z', 'redshift']
    df.columns = column_names
    new_column_order = ['mag_u', 'mag_g', 'mag_r', 'mag_i', 'mag_z', 'mag_y', 'redshift']
    df = df[new_column_order]
    return df

Save bliss predicted mag & redshift

In [ ]:
df_bliss = pd.DataFrame(torch.cat([bliss_mag, true_redshift], dim=1))
df_bliss = transform_column(df_bliss)


In [ ]:
dataset_name = "bliss_pred_mag"
path = f"{home}/data/redshift/dc2/{dataset_name}.pkl"
df_bliss.to_pickle(path)

In [ ]:
df_bliss.describe()

Save true mag & redshift

In [ ]:
df_true = pd.DataFrame(torch.cat([true_mag, true_redshift], dim=1))
df_true = transform_column(df_true)

In [ ]:
dataset_name = "bliss_true_mag"
path = f"{home}/data/redshift/dc2/{dataset_name}.pkl"
df_true.to_pickle(path)

In [ ]:
df_true.describe()